# Laboratorium 10 – Powiększanie obrazów i interpolacja funkcjami sklejanymi B-Spline

**Autor:** mgr inż. Krzysztof Borkowski  
**Przedmiot:** Techniki Wizyjne i Przetwarzanie Obrazów

## 1. Teoria

### 1.1. Wprowadzenie do powiększania obrazów

Powiększanie obrazów (ang. *image upscaling*, *image enlargement*) to proces zwiększania rozdzielczości obrazu poprzez dodanie nowych pikseli. Kluczowym wyzwaniem jest **interpolacja** – oszacowanie wartości nowych pikseli na podstawie istniejących.

**Podstawowe metody interpolacji:**
- **Najbliższy sąsiad** (nearest neighbor) – szybka, ale powoduje efekt "schodkowania"
- **Interpolacja liniowa** – lepsza jakość, ale wciąż widoczne artefakty
- **Interpolacja kubiczna** – wysoka jakość, gładkie przejścia
- **Funkcje sklejane B-Spline** – elastyczne, parametryzowalne, wysokiej jakości

### 1.2. Funkcje sklejane B-Spline

**Funkcje B-Spline** (Basis Spline) to rodzina funkcji bazowych używanych do interpolacji i aproksymacji. Charakteryzują się:
- **Lokalnym wsparciem** – zmiana w jednym punkcie wpływa tylko na ograniczony obszar
- **Gładkością** – funkcje wyższych rzędów są bardziej gładkie
- **Efektywnością obliczeniową** – można je obliczać rekurencyjnie

### 1.3. Definicje matematyczne funkcji B-Spline

Funkcje B-Spline definiujemy dla parametru $t \in [0, 1]$, który reprezentuje pozycję między dwoma pikselami.

#### **B-Spline rzędu 0 (stała, nearest neighbor):**

$$
B^0(t) = \begin{cases}
1 & \text{dla } t \in [0, 1) \\
0 & \text{w przeciwnym przypadku}
\end{cases}
$$

**Interpretacja:** Wybiera wartość najbliższego piksela (skok jednostkowy).

#### **B-Spline rzędu 1 (liniowa):**

$$
B^1(t) = \begin{cases}
t & \text{dla } t \in [0, 1) \\
2 - t & \text{dla } t \in [1, 2) \\
0 & \text{w przeciwnym przypadku}
\end{cases}
$$

**Interpretacja:** Liniowa interpolacja między dwoma pikselami (funkcja "namiotowa").

#### **B-Spline rzędu 2 (kwadratowa):**

$$
B^2(t) = \begin{cases}
\frac{1}{2}t^2 & \text{dla } t \in [0, 1) \\
-t^2 + 3t - \frac{3}{2} & \text{dla } t \in [1, 2) \\
\frac{1}{2}(3-t)^2 & \text{dla } t \in [2, 3) \\
0 & \text{w przeciwnym przypadku}
\end{cases}
$$

**Interpretacja:** Gładka interpolacja kwadratowa, ciągła pierwsza pochodna.

#### **B-Spline rzędu 3 (kubiczna):**

$$
B^3(t) = \begin{cases}
\frac{1}{6}t^3 & \text{dla } t \in [0, 1) \\
-\frac{1}{2}t^3 + 2t^2 - 2t + \frac{2}{3} & \text{dla } t \in [1, 2) \\
\frac{1}{2}t^3 - 4t^2 + 10t - \frac{22}{3} & \text{dla } t \in [2, 3) \\
\frac{1}{6}(4-t)^3 & \text{dla } t \in [3, 4) \\
0 & \text{w przeciwnym przypadku}
\end{cases}
$$

**Interpretacja:** Bardzo gładka interpolacja kubiczna, ciągła druga pochodna. **Najlepsza jakość** dla większości zastosowań.

### 1.4. Wizualizacja funkcji B-Spline

```
B⁰(t):  ┌────┐
        │    │
    ────┘    └────
        0    1

B¹(t):    /\
         /  \
    ────/    \────
        0  1  2

B²(t):     ╱‾‾╲
          ╱    ╲
    ─────╱      ╲─────
        0   1   2   3

B³(t):     ╱‾‾‾‾‾╲
          ╱       ╲
    ─────╱         ╲──────
        0   1   2   3   4
```

**Obserwacje:**
- Wyższy rząd → szersze wsparcie (więcej sąsiednich pikseli wpływa na wynik)
- Wyższy rząd → gładsza funkcja (lepsza jakość wizualna)
- Wyższy rząd → większy koszt obliczeniowy

### 1.5. Algorytm powiększania obrazu z B-Spline

**Krok 1:** Wybierz współczynnik powiększenia $k$ (np. $k=2$ dla podwojenia rozmiaru)

**Krok 2:** Dla każdego nowego piksela $(x_{new}, y_{new})$ w powiększonym obrazie:

1. Oblicz odpowiadającą pozycję w oryginalnym obrazie:
   $$x_{orig} = \frac{x_{new}}{k}, \quad y_{orig} = \frac{y_{new}}{k}$$

2. Wyznacz indeksy całkowite i ułamkowe:
   $$i = \lfloor x_{orig} \rfloor, \quad j = \lfloor y_{orig} \rfloor$$
   $$t_x = x_{orig} - i, \quad t_y = y_{orig} - j$$

3. Dla interpolacji 1D (np. w kierunku $x$):
   $$I_{new}(x_{new}, y) = \sum_{m} I_{orig}(i+m, j) \cdot B^n(|t_x - m|)$$
   gdzie $n$ to rząd B-Spline, a suma po $m$ obejmuje sąsiednie piksele w zasięgu funkcji.

4. Dla interpolacji 2D (separowalna):
   - Najpierw interpoluj wiersze (kierunek $x$)
   - Następnie interpoluj kolumny (kierunek $y$) na wynikach z kroku 1

**Krok 3:** Obsłuż brzegi obrazu (np. przez odbicie, powtórzenie lub zerowanie).

### 1.6. Przykład numeryczny (B-Spline rzędu 1)

Mamy piksele: $I[0]=100$, $I[1]=200$. Chcemy obliczyć wartość w $x=0.3$:

$$
I(0.3) = I[0] \cdot B^1(0.3) + I[1] \cdot B^1(1-0.3)
$$
$$
= 100 \cdot 0.7 + 200 \cdot 0.3 = 70 + 60 = 130
$$

**Metoda Feynmana:** Wyobraź sobie, że idziesz od piksela 0 do piksela 1. W 30% drogi jesteś bliżej piksela 0 (waga 70%), więc jego wpływ jest większy.

### 1.7. Zalety i wady różnych rzędów B-Spline

| Rząd | Zalety | Wady |
|------|--------|------|
| **0** | Bardzo szybka, prosta implementacja | Efekt "schodkowania", brak gładkości |
| **1** | Szybka, prosta, lepsza od rzędu 0 | Widoczne krawędzie, brak gładkości C¹ |
| **2** | Gładka (C¹), dobry kompromis | Rzadko używana (rząd 3 niewiele wolniejszy) |
| **3** | Bardzo gładka (C²), najlepsza jakość | Wolniejsza, wymaga 4 pikseli w każdym kierunku |

**Wniosek:** W praktyce najczęściej używa się **B-Spline rzędu 3** (kubicznej) jako optymalnego kompromisu między jakością a szybkością.

## 2. Kod startowy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import urllib.request
import cv2
import time

### Funkcje pomocnicze

In [ ]:
def rgb_to_gray(img):
    """Konwersja RGB -> skala szarości (wzór ITU-R BT.601)"""
    if len(img.shape) == 2:
        return img
    return np.dot(img[...,:3], [0.299, 0.587, 0.114]).astype(np.uint8)

def display_images(images, titles=None, cmap='gray', figsize=(15, 5)):
    """Wyświetlanie wielu obrazów obok siebie"""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for i, (img, ax) in enumerate(zip(images, axes)):
        ax.imshow(img, cmap=cmap)
        ax.axis('off')
        if titles:
            ax.set_title(titles[i])
    plt.tight_layout()
    plt.show()

def normalize_image(img):
    """Normalizacja obrazu do zakresu [0, 255]"""
    img_min, img_max = img.min(), img.max()
    if img_max - img_min == 0:
        return img.astype(np.uint8)
    return ((img - img_min) / (img_max - img_min) * 255).astype(np.uint8)

### Wczytanie obrazów testowych

In [ ]:
# Obraz Lena (klasyczny obraz testowy)
url_lena = 'https://raw.githubusercontent.com/kbor89/TWiPO/main/images/lenna.png'
urllib.request.urlretrieve(url_lena, 'lena.png')
img_lena_color = np.array(Image.open('lena.png'))
img_lena = rgb_to_gray(img_lena_color).astype(np.uint8)

# Zmniejszenie obrazu do 100x100
img_lenna_gray = np.array(Image.fromarray(img_lena).resize((100, 100), Image.LANCZOS))

print(f"Lena - rozmiar: {img_lena.shape}, typ: {img_lena.dtype}")
print(f"lenna_gray - rozmiar: {img_lenna_gray.shape}, typ: {img_lenna_gray.dtype}")
display_images([img_lena_color, img_lena, img_lenna_gray], ['Lena (kolor)', 'Lena (szarości)', 'Lena (szarości + zmniejszony)'])

### Prosty obraz testowy (szachownica)

In [ ]:
# Szachownica 8x8 do testowania interpolacji
checkerboard = np.zeros((8, 8), dtype=np.uint8)
checkerboard[::2, ::2] = 255
checkerboard[1::2, 1::2] = 255

# Prosty gradient
gradient = np.linspace(0, 255, 8).astype(np.uint8)
gradient = np.tile(gradient, (8, 1))

display_images([checkerboard, gradient], ['Szachownica 8x8', 'Gradient 8x8'])

## 3. Zadania do wykonania

### Zadanie 1: Wybór i wyświetlenie fragmentu obrazu (ROI)

**Cel:** Zaimplementuj funkcję `select_roi(img, x, y, width, height)`, która wycina prostokątny fragment obrazu.

**Parametry:**
- `img` – obraz wejściowy (numpy array)
- `x, y` – współrzędne lewego górnego rogu ROI
- `width, height` – szerokość i wysokość ROI

**Zwraca:**
- Fragment obrazu jako numpy array

**Wskazówki:**
- Użyj indeksowania numpy: `img[y:y+height, x:x+width]`
- Sprawdź, czy ROI nie wykracza poza granice obrazu
- Pamiętaj: w numpy pierwsza współrzędna to wiersz (y), druga to kolumna (x)

**Pytania kontrolne:**
1. Dlaczego w numpy używamy `img[y, x]` zamiast `img[x, y]`?
2. Co się stanie, jeśli ROI wykroczy poza obraz?

**Test:** Wytnij fragment twarzy Leny (np. oko) i wyświetl go.

In [ ]:
# TODO: Zaimplementuj funkcję select_roi
def select_roi(img, x, y, width, height):
    # Twój kod tutaj
    pass

# Test
roi = select_roi(img_lena, 250, 250, 100, 100)
display_images([img_lena, roi], ['Obraz oryginalny', 'ROI (100x100)'])

### Zadanie 2: Implementacja funkcji B-Spline rzędu 0 (stała)

**Cel:** Zaimplementuj funkcję `bspline_0(t)` obliczającą wartość B-Spline rzędu 0 dla parametru `t`.

**Definicja matematyczna:**
$$
B^0(t) = \begin{cases}
1 & \text{dla } t \in [0, 1) \\
0 & \text{w przeciwnym przypadku}
\end{cases}
$$

**Parametry:**
- `t` – parametr (float lub numpy array), wartość w przedziale [0, 4)

**Zwraca:**
- Wartość funkcji B-Spline rzędu 0

**Wskazówki:**
- Użyj instrukcji warunkowych lub `np.where()` dla tablic
- Funkcja powinna działać zarówno dla pojedynczych wartości, jak i tablic numpy

**Pytania kontrolne:**
1. Jaki jest zakres wsparcia funkcji B⁰(t)? (dla jakich t funkcja jest niezerowa?)
2. Czy funkcja B⁰(t) jest ciągła?

**Test:** Narysuj wykres funkcji dla t ∈ [-1, 2]

In [ ]:
# TODO: Zaimplementuj funkcję bspline_0
def bspline_0(t):
    # Twój kod tutaj
    pass

# Test - wykres
t_vals = np.linspace(-1, 2, 300)
b0_vals = bspline_0(t_vals)
plt.figure(figsize=(8, 4))
plt.plot(t_vals, b0_vals, linewidth=2)
plt.grid(True, alpha=0.3)
plt.xlabel('t')
plt.ylabel('B⁰(t)')
plt.title('Funkcja B-Spline rzędu 0')
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.show()

### Zadanie 3: Implementacja funkcji B-Spline rzędu 1 (liniowa)

**Cel:** Zaimplementuj funkcję `bspline_1(t)` obliczającą wartość B-Spline rzędu 1.

**Definicja matematyczna:**
$$
B^1(t) = \begin{cases}
t & \text{dla } t \in [0, 1) \\
2 - t & \text{dla } t \in [1, 2) \\
0 & \text{w przeciwnym przypadku}
\end{cases}
$$

**Parametry:**
- `t` – parametr (float lub numpy array)

**Zwraca:**
- Wartość funkcji B-Spline rzędu 1

**Wskazówki:**
- Funkcja ma kształt "namiotu" (trójkąta)
- Maksimum wynosi 1.0 w t=1
- Użyj `np.where()` lub instrukcji warunkowych

**Pytania kontrolne:**
1. Jaki jest zakres wsparcia funkcji B¹(t)?
2. Ile sąsiednich pikseli wpływa na interpolację przy użyciu B¹?
3. Czy funkcja B¹(t) jest ciągła? A jej pochodna?

**Test:** Narysuj wykres funkcji dla t ∈ [-1, 3]

In [ ]:
# TODO: Zaimplementuj funkcję bspline_1
def bspline_1(t):
    # Twój kod tutaj
    pass

# Test - wykres
t_vals = np.linspace(-1, 3, 400)
b1_vals = bspline_1(t_vals)
plt.figure(figsize=(8, 4))
plt.plot(t_vals, b1_vals, linewidth=2)
plt.grid(True, alpha=0.3)
plt.xlabel('t')
plt.ylabel('B¹(t)')
plt.title('Funkcja B-Spline rzędu 1')
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.show()

### Zadanie 4: Implementacja funkcji B-Spline rzędu 3 (kubiczna)

**Cel:** Zaimplementuj funkcję `bspline_3(t)` obliczającą wartość B-Spline rzędu 3.

**Definicja matematyczna:**
$$
B^3(t) = \begin{cases}
\frac{1}{6}t^3 & \text{dla } t \in [0, 1) \\
-\frac{1}{2}t^3 + 2t^2 - 2t + \frac{2}{3} & \text{dla } t \in [1, 2) \\
\frac{1}{2}t^3 - 4t^2 + 10t - \frac{22}{3} & \text{dla } t \in [2, 3) \\
\frac{1}{6}(4-t)^3 & \text{dla } t \in [3, 4) \\
0 & \text{w przeciwnym przypadku}
\end{cases}
$$

**Parametry:**
- `t` – parametr (float lub numpy array)

**Zwraca:**
- Wartość funkcji B-Spline rzędu 3

**Wskazówki:**
- Funkcja ma 4 przedziały
- Użyj `np.where()` lub instrukcji warunkowych dla każdego przedziału
- Sprawdź poprawność implementacji porównując z wykresem z teorii
- Zwróć uwagę na ułamki: 2/3 ≈ 0.6667, 22/3 ≈ 7.3333

**Pytania kontrolne:**
1. Jaki jest zakres wsparcia funkcji B³(t)?
2. Ile sąsiednich pikseli wpływa na interpolację przy użyciu B³?
3. Dlaczego B³ daje lepszą jakość niż B¹?

**Test:** Narysuj wykres funkcji dla t ∈ [-1, 5] i porównaj z B⁰ i B¹

In [ ]:
# TODO: Zaimplementuj funkcję bspline_3
def bspline_3(t):
    # Twój kod tutaj
    pass

# Test - wykres porównawczy
t_vals = np.linspace(-1, 5, 600)
b0_vals = bspline_0(t_vals)
b1_vals = bspline_1(t_vals)
b3_vals = bspline_3(t_vals)

plt.figure(figsize=(12, 4))
plt.plot(t_vals, b0_vals, label='B⁰(t) - stała', linewidth=2)
plt.plot(t_vals, b1_vals, label='B¹(t) - liniowa', linewidth=2)
plt.plot(t_vals, b3_vals, label='B³(t) - kubiczna', linewidth=2)
plt.grid(True, alpha=0.3)
plt.xlabel('t')
plt.ylabel('B(t)')
plt.title('Porównanie funkcji B-Spline różnych rzędów')
plt.legend()
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.show()

### Zadanie 5: Powiększanie obrazu 1D z B-Spline

**Cel:** Zaimplementuj funkcję `enlarge_1d(row, scale, bspline_func)`, która powiększa jednowymiarowy sygnał (wiersz obrazu) używając wybranej funkcji B-Spline.

**Parametry:**
- `row` – jednowymiarowa tablica (wiersz obrazu)
- `scale` – współczynnik powiększenia (np. 2 dla podwojenia)
- `bspline_func` – funkcja B-Spline (np. `bspline_1` lub `bspline_3`)

**Zwraca:**
- Powiększony wiersz

**Algorytm:**
1. Oblicz nowy rozmiar: `new_size = len(row) * scale`
2. Dla każdego nowego piksela `i` w `[0, new_size)`:
   - Oblicz pozycję w oryginalnym wierszu: `x_orig = i / scale`
   - Wyznacz indeks całkowity: `x_int = floor(x_orig)`
   - Wyznacz część ułamkową: `t = x_orig - x_int`
   - Oblicz wartość jako sumę ważoną sąsiednich pikseli:
     $$\text{new\_pixel}[i] = \sum_{k} \text{row}[x\_int + k - \text{offset}] \cdot B(|t - k|)$$
     gdzie offset zależy od rzędu B-Spline (0→0, 1→0, 3→1)
3. Obsłuż brzegi (np. przez `np.clip` dla indeksów)

**Wskazówki:**
- Dla B⁰: offset=0.5, zakres k=[0]
- Dla B¹: offset=1, zakres k=[0, 1]
- Dla B³: offset=2, zakres k=[-1, 0, 1, 2]
- Użyj `np.clip(index, 0, len(row)-1)` do obsługi brzegów

**Pytania kontrolne:**
1. Dlaczego dla B³ używamy offset=2?
2. Co się stanie, jeśli nie obsłużymy brzegów?

**Test:** Powiększ prosty gradient 2x używając B¹ i B³

In [ ]:
# TODO: Zaimplementuj funkcję enlarge_1d
def enlarge_1d(row, scale, bspline_func):
    # Twój kod tutaj
    pass

# Test - gradient
test_row = np.array([0, 64, 128, 192, 255], dtype=float)
enlarged_b1 = enlarge_1d(test_row, 2, bspline_1)
enlarged_b3 = enlarge_1d(test_row, 2, bspline_3)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(test_row, 'o-', linewidth=2, markersize=8)
plt.title('Oryginalny (5 pikseli)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(enlarged_b1, 'o-', linewidth=2, markersize=6)
plt.title('B¹ - powiększony 2x (10 pikseli)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(enlarged_b3, 'o-', linewidth=2, markersize=6)
plt.title('B³ - powiększony 2x (10 pikseli)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Zadanie 6: Powiększanie obrazu 2D z B-Spline (separowalne)

**Cel:** Zaimplementuj funkcję `enlarge_image(img, scale, bspline_func)`, która powiększa obraz 2D używając separowalnej interpolacji B-Spline.

**Parametry:**
- `img` – obraz wejściowy (2D numpy array)
- `scale` – współczynnik powiększenia
- `bspline_func` – funkcja B-Spline

**Zwraca:**
- Powiększony obraz

**Algorytm (separowalny):**
1. **Krok 1:** Powiększ wszystkie wiersze obrazu (interpolacja w kierunku X)
   - Dla każdego wiersza `i`: `temp[i, :] = enlarge_1d(img[i, :], scale, bspline_func)`
2. **Krok 2:** Powiększ wszystkie kolumny wyniku z kroku 1 (interpolacja w kierunku Y)
   - Dla każdej kolumny `j`: `result[:, j] = enlarge_1d(temp[:, j], scale, bspline_func)`

**Wskazówki:**
- Użyj funkcji `enlarge_1d` z poprzedniego zadania
- Interpolacja separowalna: najpierw wiersze, potem kolumny
- Pamiętaj o konwersji wyniku do `uint8` na końcu

**Pytania kontrolne:**
1. Dlaczego interpolacja separowalna jest efektywna obliczeniowo?
2. Jaka jest złożoność obliczeniowa powiększenia obrazu NxN o współczynnik k?

**Test:** Powiększ szachownicę 8x8 do 64x64 używając B⁰, B¹ i B³

In [ ]:
# TODO: Zaimplementuj funkcję enlarge_image
def enlarge_image(img, scale, bspline_func):
    # Twój kod tutaj
    pass

# Test - szachownica
enlarged_b0 = enlarge_image(checkerboard, 8, bspline_0)
enlarged_b1 = enlarge_image(checkerboard, 8, bspline_1)
enlarged_b3 = enlarge_image(checkerboard, 8, bspline_3)

display_images(
    [checkerboard, enlarged_b0, enlarged_b1, enlarged_b3],
    ['Oryginał 8x8', 'B⁰ (64x64)', 'B¹ (64x64)', 'B³ (64x64)'],
    figsize=(16, 4)
)

### Zadanie 7: Porównanie metod interpolacji na obrazie Leny

**Cel:** Porównaj wizualnie i ilościowo różne metody interpolacji B-Spline na fragmencie obrazu Leny.

**Zadania:**
1. Wytnij fragment obrazu Leny (np. 50x50 pikseli z twarzy)
2. Powiększ go 4x używając:
   - B-Spline rzędu 0 (nearest neighbor)
   - B-Spline rzędu 1 (liniowa)
   - B-Spline rzędu 3 (kubiczna)
   - OpenCV `cv2.resize()` z `INTER_CUBIC` (dla porównania)
3. Wyświetl wszystkie wyniki obok siebie
4. Oblicz i porównaj:
   - Czas wykonania każdej metody
   - MSE (Mean Squared Error) względem metody OpenCV

**Wskazówki:**
- Użyj `time.time()` do pomiaru czasu
- MSE: `np.mean((img1.astype(float) - img2.astype(float))**2)`
- OpenCV: `cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)`

**Pytania kontrolne:**
1. Która metoda daje najlepszą jakość wizualną?
2. Która metoda jest najszybsza?
3. Dlaczego OpenCV jest szybsze od Twojej implementacji?

In [ ]:
# TODO: Zaimplementuj porównanie metod
# 1. Wytnij ROI z obrazu Leny
roi_lena = select_roi(img_lena, 200, 200, 50, 50)

# 2. Powiększ używając różnych metod i zmierz czas
scale = 4

# B-Spline rzędu 0
start = time.time()
# Twój kod tutaj
enlarged_b0_lena = None
time_b0 = time.time() - start

# B-Spline rzędu 1
start = time.time()
# Twój kod tutaj
enlarged_b1_lena = None
time_b1 = time.time() - start

# B-Spline rzędu 3
start = time.time()
# Twój kod tutaj
enlarged_b3_lena = None
time_b3 = time.time() - start

# OpenCV INTER_CUBIC
start = time.time()
# Twój kod tutaj
enlarged_cv_lena = None
time_cv = time.time() - start

# 3. Wyświetl wyniki
display_images(
    [roi_lena, enlarged_b0_lena, enlarged_b1_lena, enlarged_b3_lena, enlarged_cv_lena],
    ['Oryginał 50x50', 'B⁰', 'B¹', 'B³', 'OpenCV Cubic'],
    figsize=(18, 4)
)

# 4. Oblicz MSE względem OpenCV
# Twój kod tutaj
print(f"\nCzasy wykonania:")
print(f"B⁰: {time_b0:.4f}s")
print(f"B¹: {time_b1:.4f}s")
print(f"B³: {time_b3:.4f}s")
print(f"OpenCV: {time_cv:.4f}s")

print(f"\nMSE względem OpenCV:")
# Wyświetl MSE tutaj

In [ ]:
# ROZWIĄZANIE Zadanie 7
# 1. Wytnij ROI z obrazu Leny
# roi_lena = select_roi(img_lena, 200, 200, 50, 50)
roi_lena = select_roi(img_lenna_gray, 46, 47, 10, 10)

# 2. Powiększ używając różnych metod i zmierz czas
scale = 4

# B-Spline rzędu 0
start = time.time()
enlarged_b0_lena = enlarge_image(roi_lena, scale, bspline_0)
time_b0 = time.time() - start

# B-Spline rzędu 1
start = time.time()
enlarged_b1_lena = enlarge_image(roi_lena, scale, bspline_1)
time_b1 = time.time() - start

# B-Spline rzędu 3
start = time.time()
enlarged_b3_lena = enlarge_image(roi_lena, scale, bspline_3)
time_b3 = time.time() - start

# OpenCV INTER_CUBIC
start = time.time()
new_size = (roi_lena.shape[1] * scale, roi_lena.shape[0] * scale)
enlarged_cv_lena = cv2.resize(roi_lena, new_size, interpolation=cv2.INTER_CUBIC)
time_cv = time.time() - start

# 3. Wyświetl wyniki
display_images(
    [roi_lena, enlarged_b0_lena, enlarged_b1_lena, enlarged_b3_lena, enlarged_cv_lena],
    ['Oryginał 50x50', 'B⁰', 'B¹', 'B³', 'OpenCV Cubic'],
    figsize=(18, 4)
)

# 4. Oblicz MSE względem OpenCV
def calculate_mse(img1, img2):
    return np.mean((img1.astype(float) - img2.astype(float))**2)

mse_b0 = calculate_mse(enlarged_b0_lena, enlarged_cv_lena)
mse_b1 = calculate_mse(enlarged_b1_lena, enlarged_cv_lena)
mse_b3 = calculate_mse(enlarged_b3_lena, enlarged_cv_lena)

print(f"\nCzasy wykonania:")
print(f"B⁰: {time_b0:.4f}s")
print(f"B¹: {time_b1:.4f}s")
print(f"B³: {time_b3:.4f}s")
print(f"OpenCV: {time_cv:.4f}s")
print(f"\nPrzyśpieszenie OpenCV względem B³: {time_b3/time_cv:.1f}x")

print(f"\nMSE względem OpenCV INTER_CUBIC:")
print(f"B⁰: {mse_b0:.2f}")
print(f"B¹: {mse_b1:.2f}")
print(f"B³: {mse_b3:.2f}")

# Odpowiedzi na pytania kontrolne:
# 1. B³ (kubiczna) daje najlepszą jakość wizualną - gładkie przejścia, brak artefaktów
# 2. OpenCV jest najszybsze (zoptymalizowana implementacja w C++)
# 3. OpenCV jest szybsze, bo:
#    - Implementacja w C++ (vs Python)
#    - Optymalizacje SIMD (SSE, AVX)
#    - Wielowątkowość
#    - Cache-friendly memory access

## 4. Przykłady zaawansowane

### Przykład 1: Powiększanie całego obrazu Leny

Powiększenie całego obrazu Leny 2x używając B-Spline kubicznej.

In [ ]:
# Powiększenie całego obrazu Leny 2x
print("Powiększanie obrazu Leny 100x100 -> 400x400...")
start = time.time()
lena_enlarged = enlarge_image(img_lenna_gray, 3, bspline_3)
elapsed = time.time() - start
print(f"Czas: {elapsed:.2f}s")

display_images(
    [img_lena, img_lenna_gray, lena_enlarged],
    [f'Oryginał {img_lena.shape}',f'Pomniejszony {img_lenna_gray.shape}', f'Powiększony 2x {lena_enlarged.shape}'],
    figsize=(12, 6)
)

### Przykład 2: Porównanie wszystkich metod na szachownicy

Wizualizacja różnic między metodami interpolacji na obrazie testowym.

In [ ]:
# Powiększenie szachownicy różnymi metodami
scale = 16
checker_b0 = enlarge_image(checkerboard, scale, bspline_0)
checker_b1 = enlarge_image(checkerboard, scale, bspline_1)
checker_b3 = enlarge_image(checkerboard, scale, bspline_3)
checker_cv = cv2.resize(checkerboard, (checkerboard.shape[1]*scale, checkerboard.shape[0]*scale), 
                        interpolation=cv2.INTER_CUBIC)

# Wyświetl fragment (środek obrazu)
crop_size = 64
center = checker_b0.shape[0] // 2
crop_slice = (slice(center-crop_size//2, center+crop_size//2), 
              slice(center-crop_size//2, center+crop_size//2))

display_images(
    [checker_b0[crop_slice], checker_b1[crop_slice], 
     checker_b3[crop_slice], checker_cv[crop_slice]],
    ['B⁰ (Nearest)', 'B¹ (Linear)', 'B³ (Cubic)', 'OpenCV Cubic'],
    figsize=(16, 4)
)

print("Obserwacje:")
print("- B⁰: Wyraźne 'schodki' na krawędziach")
print("- B¹: Liniowe przejścia, wciąż widoczne artefakty")
print("- B³: Gładkie przejścia, najlepsza jakość")
print("- OpenCV: Podobna jakość do B³, ale szybsza")

### Przykład 3: Analiza wpływu współczynnika powiększenia

Sprawdzenie jak współczynnik powiększenia wpływa na jakość i czas obliczeń.

In [ ]:
# Test różnych współczynników powiększenia
test_img = select_roi(img_lena, 200, 200, 32, 32)
scales = [2, 4, 8]

results = []
times = []

for scale in scales:
    start = time.time()
    enlarged = enlarge_image(test_img, scale, bspline_3)
    elapsed = time.time() - start
    results.append(enlarged)
    times.append(elapsed)
    print(f"Scale {scale}x: {test_img.shape} -> {enlarged.shape}, czas: {elapsed:.4f}s")

display_images(
    [test_img] + results,
    ['Oryginał 32x32'] + [f'{scale}x ({r.shape[0]}x{r.shape[1]})' for scale, r in zip(scales, results)],
    figsize=(16, 4)
)

# Wykres czasu vs współczynnik powiększenia
plt.figure(figsize=(8, 4))
plt.plot(scales, times, 'o-', linewidth=2, markersize=8)
plt.xlabel('Współczynnik powiększenia')
plt.ylabel('Czas [s]')
plt.title('Czas obliczeń vs współczynnik powiększenia (B³)')
plt.grid(True, alpha=0.3)
plt.show()

### Przykład 4: Wizualizacja wag interpolacji

Zobaczmy jak wyglądają wagi dla różnych funkcji B-Spline przy interpolacji konkretnego piksela.

In [ ]:
# Wizualizacja wag dla t=0.3 (30% drogi między pikselami)
t_test = 0.3

# B¹: wagi dla 2 sąsiadów
weights_b1 = [bspline_1(abs(t_test - k)) for k in [0, 1]]
print(f"B¹ dla t={t_test}:")
print(f"  Piksel[0]: waga = {weights_b1[0]:.3f}")
print(f"  Piksel[1]: waga = {weights_b1[1]:.3f}")
print(f"  Suma wag: {sum(weights_b1):.3f}")

# B³: wagi dla 4 sąsiadów
weights_b3 = [bspline_3(abs(t_test - k + 1)) for k in [-1, 0, 1, 2]]
print(f"\nB³ dla t={t_test}:")
for i, w in enumerate(weights_b3):
    print(f"  Piksel[{i-1}]: waga = {w:.3f}")
print(f"  Suma wag: {sum(weights_b3):.3f}")

# Wykres wag
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar([0, 1], weights_b1, color=['blue', 'orange'])
ax1.set_xlabel('Indeks piksela')
ax1.set_ylabel('Waga')
ax1.set_title(f'Wagi B¹ dla t={t_test}')
ax1.set_ylim([0, 1])
ax1.grid(True, alpha=0.3)

ax2.bar([-1, 0, 1, 2], weights_b3, color=['red', 'blue', 'orange', 'green'])
ax2.set_xlabel('Indeks piksela')
ax2.set_ylabel('Waga')
ax2.set_title(f'Wagi B³ dla t={t_test}')
ax2.set_ylim([0, 1])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nWniosek: Suma wag zawsze wynosi ~1.0 (zachowanie jasności)")

### Przykład 5: Porównanie z innymi metodami OpenCV

Porównanie B-Spline z wszystkimi metodami dostępnymi w OpenCV.

In [ ]:
# Porównanie z metodami OpenCV
test_roi = select_roi(img_lena, 220, 220, 40, 40)
scale = 5
new_size = (test_roi.shape[1] * scale, test_roi.shape[0] * scale)

# Metody OpenCV
cv_nearest = cv2.resize(test_roi, new_size, interpolation=cv2.INTER_NEAREST)
cv_linear = cv2.resize(test_roi, new_size, interpolation=cv2.INTER_LINEAR)
cv_cubic = cv2.resize(test_roi, new_size, interpolation=cv2.INTER_CUBIC)
cv_lanczos = cv2.resize(test_roi, new_size, interpolation=cv2.INTER_LANCZOS4)

# Nasze metody
our_b0 = enlarge_image(test_roi, scale, bspline_0)
our_b1 = enlarge_image(test_roi, scale, bspline_1)
our_b3 = enlarge_image(test_roi, scale, bspline_3)

# Wyświetl
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

images = [test_roi, cv_nearest, cv_linear, cv_cubic, cv_lanczos, our_b0, our_b1, our_b3]
titles = ['Oryginał', 'CV Nearest', 'CV Linear', 'CV Cubic', 'CV Lanczos4',
          'Nasza B⁰', 'Nasza B¹', 'Nasza B³']

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Obserwacje:")
print("- INTER_NEAREST ≈ nasza B⁰")
print("- INTER_LINEAR ≈ nasza B¹")
print("- INTER_CUBIC ≈ nasza B³ (ale OpenCV używa innej wersji kubicznej)")
print("- INTER_LANCZOS4 - najlepsza jakość, ale najwolniejsza")

## Podsumowanie

### Czego się nauczyliśmy:

1. **Funkcje B-Spline** to elastyczne narzędzie do interpolacji obrazów
2. **Wyższy rząd** = lepsza jakość, ale większy koszt obliczeniowy
3. **B³ (kubiczna)** to najlepszy kompromis dla większości zastosowań
4. **Interpolacja separowalna** (wiersze → kolumny) jest efektywna obliczeniowo
5. **Implementacje biblioteczne** (OpenCV) są znacznie szybsze dzięki optymalizacjom

### Kluczowe wnioski:

| Metoda | Jakość | Szybkość | Zastosowanie |
|--------|--------|----------|-------------|
| **B⁰** | Niska | Bardzo szybka | Gdy szybkość > jakość |
| **B¹** | Średnia | Szybka | Prosty kompromis |
| **B³** | Wysoka | Średnia | Większość zastosowań |
| **Lanczos** | Najwyższa | Wolna | Gdy jakość > szybkość |

### Dalsze kierunki:

- **Super-resolution** z użyciem sieci neuronowych (SRCNN, ESRGAN)
- **Adaptacyjna interpolacja** (różne metody dla różnych regionów)
- **Interpolacja z zachowaniem krawędzi** (edge-directed interpolation)
- **Optymalizacja kodu** (Numba, Cython, CUDA)

---

**Koniec laboratorium 9**